# Interactive Exercise: Select the Best-Fitting ARIMA Model

In this exercise, you will estimate and compare several candidate ARIMA models.

You will:

1. Select an expenditure category.
2. Determine the differencing order.
3. Estimate manual candidate models.
4. Use `auto.arima()` as a benchmark.
5. Compare AIC, AICc, and BIC.
6. Review residual diagnostics.
7. Select and save the final model.

> **About the data:** The dataset is simulated for instructional purposes and does not contain official Illinois monthly expenditure figures.


## 1. Load the Required Packages


In [ ]:
packages <- c(
  "readr",
  "dplyr",
  "forecast",
  "tseries",
  "tibble"
)

installed <- rownames(installed.packages())
missing_packages <- packages[!(packages %in% installed)]

if (length(missing_packages) > 0) {
  install.packages(
    missing_packages,
    repos = "https://cloud.r-project.org"
  )
}

invisible(
  lapply(
    packages,
    library,
    character.only = TRUE
  )
)

## 2. Load the Simulated Monthly Expenditure Data


In [ ]:
df <- read_csv(
  "../data/illinois_monthly_expenditures_simulated.csv"
)

df

## 3. Select an Expenditure Category


In [ ]:
selected_series <- "education"

## 4. Prepare the Monthly Time Series


In [ ]:
plot_df <- df %>%
  filter(series == selected_series) %>%
  mutate(date = as.Date(date)) %>%
  arrange(date)

if (nrow(plot_df) == 0) {
  stop("The selected expenditure series was not found.")
}

start_year <- as.integer(format(min(plot_df$date), "%Y"))
start_month <- as.integer(format(min(plot_df$date), "%m"))

expenditure_ts <- ts(
  plot_df$expenditure,
  start = c(start_year, start_month),
  frequency = 12
)

expenditure_ts

## 5. Determine the Differencing Order


In [ ]:
suggested_d <- forecast::ndiffs(
  expenditure_ts,
  test = "adf"
)

cat(
  "Suggested differencing order: d =",
  suggested_d
)

## 6. Define Candidate Models

These models correspond to the common candidates identified from the ACF and PACF plots.

Edit the values if your plots suggest different specifications.


In [ ]:
candidate_orders <- tibble(
  model_name = c(
    "AR candidate",
    "MA candidate",
    "Mixed candidate"
  ),
  p = c(1, 0, 1),
  d = c(suggested_d, suggested_d, suggested_d),
  q = c(0, 1, 1)
)

candidate_orders

## 7. Estimate the Candidate Models


In [ ]:
model_ar <- forecast::Arima(
  expenditure_ts,
  order = c(
    candidate_orders$p[1],
    candidate_orders$d[1],
    candidate_orders$q[1]
  )
)

model_ma <- forecast::Arima(
  expenditure_ts,
  order = c(
    candidate_orders$p[2],
    candidate_orders$d[2],
    candidate_orders$q[2]
  )
)

model_mixed <- forecast::Arima(
  expenditure_ts,
  order = c(
    candidate_orders$p[3],
    candidate_orders$d[3],
    candidate_orders$q[3]
  )
)

## 8. Use Automatic Model Selection

The `auto.arima()` function searches across many candidate models and selects the specification with the lowest chosen information criterion.


In [ ]:
model_auto <- forecast::auto.arima(
  expenditure_ts,
  seasonal = FALSE,
  stepwise = FALSE,
  approximation = FALSE,
  test = "adf",
  ic = "bic"
)

model_auto

> **Reflection:** Try changing `ic = "bic"` to `ic = "aic"`. Does the selected model change? Why do you think different information criteria favor different models?

## 9. Compare Information Criteria

Lower AIC, AICc, and BIC values indicate a better balance between model fit and complexity.


In [ ]:
model_comparison <- tibble(
  model = c(
    paste0(
      "ARIMA(",
      candidate_orders$p[1], ",",
      candidate_orders$d[1], ",",
      candidate_orders$q[1], ")"
    ),
    paste0(
      "ARIMA(",
      candidate_orders$p[2], ",",
      candidate_orders$d[2], ",",
      candidate_orders$q[2], ")"
    ),
    paste0(
      "ARIMA(",
      candidate_orders$p[3], ",",
      candidate_orders$d[3], ",",
      candidate_orders$q[3], ")"
    ),
    "Automatic selection"
  ),
  AIC = c(
    AIC(model_ar),
    AIC(model_ma),
    AIC(model_mixed),
    AIC(model_auto)
  ),
  AICc = c(
    model_ar$aicc,
    model_ma$aicc,
    model_mixed$aicc,
    model_auto$aicc
  ),
  BIC = c(
    BIC(model_ar),
    BIC(model_ma),
    BIC(model_mixed),
    BIC(model_auto)
  )
) %>%
  arrange(AICc)

model_comparison

## 10. Review the Automatically Selected Model


In [ ]:
selected_order <- forecast::arimaorder(model_auto)

cat(
  "Automatically selected model: ARIMA(",
  selected_order["p"], ",",
  selected_order["d"], ",",
  selected_order["q"], ")\n",
  sep = ""
)

model_auto

## 11. Check Residual Diagnostics

A suitable model should leave residuals that resemble random noise.

The output includes:

- a residual time plot;
- a residual ACF;
- a histogram;
- a Ljung–Box test.

A Ljung–Box p-value above 0.05 indicates that there is not strong evidence of remaining residual autocorrelation.


In [ ]:
forecast::checkresiduals(model_auto)

## 12. Review Diagnostics for a Manual Candidate

You may also review the best-performing manual model.

The cell below checks the mixed candidate. Replace `model_mixed` with `model_ar` or `model_ma` if another manual model has the lower AICc.


In [ ]:
forecast::checkresiduals(model_mixed)

## 13. Compare the Automatic and Manual Results

Use the information criteria and residual diagnostics together.

Do not select a model based on one statistic alone.


In [ ]:
best_model_by_aicc <- model_comparison$model[1]

cat(
  "Lowest-AICc model:",
  best_model_by_aicc,
  "\n",
  "Automatically selected model: ARIMA(",
  selected_order["p"], ",",
  selected_order["d"], ",",
  selected_order["q"], ")",
  sep = ""
)

## 14. Select the Final Model

The example below uses the automatically selected model.

You may replace `model_auto` with a manual candidate if that model has similar information criteria, acceptable residuals, and a simpler specification.


In [ ]:
selected_model <- model_auto

final_order <- forecast::arimaorder(selected_model)

final_summary <- tibble(
  parameter = c(
    "p",
    "d",
    "q"
  ),
  meaning = c(
    "Autoregressive terms",
    "Differencing order",
    "Moving average terms"
  ),
  selected_value = c(
    final_order["p"],
    final_order["d"],
    final_order["q"]
  )
)

final_summary

## 15. Save the Selected Model


In [ ]:
dir.create(
  "../output",
  showWarnings = FALSE
)

saveRDS(
  selected_model,
  file = paste0(
    "../output/",
    selected_series,
    "_selected_arima_model.rds"
  )
)

write_csv(
  final_summary,
  paste0(
    "../output/",
    selected_series,
    "_selected_arima_parameters.csv"
  )
)

cat("The selected model and parameter summary were saved to the output folder.")

## Questions for Reflection

1. Which candidate model has the lowest AICc?

2. Does `auto.arima()` select the same specification?

3. Do the residuals fluctuate randomly around zero?

4. Does the residual ACF show meaningful remaining spikes?

5. What does the Ljung–Box test indicate?

6. Would a simpler model provide a comparable fit?

7. Which model would you select, and why?


## Try Another Expenditure Category

Return to the `selected_series` cell, choose another category, and rerun the notebook.

Compare whether different expenditure categories require different ARIMA specifications.


# Next Step

The selected model is now ready to generate forecasts and prediction intervals.
